In [126]:
import pandas as pd
import re
import numpy as np
import ast
df = pd.read_csv("naukari.csv")

## Salary Processing Pipeline

1. Read `naukari.csv` into `df`.
2. Define converters:
  - `if not disclosed`
    - then set to "Not Disclosed-Not Disclosed"
  - `convert_dollar_to_lacs(salary)`:
    - strip "$", commas
    - extract numeric values
    - multiply by 80 (INR conversion)
    - format as `low-high` or single value
  - `convert_cr_to_lacs(salary)`:
    - strip "₹"
    - extract numeric values
    - multiply by 1e7
    - format as range or single value
  - `convert_salary_value(x)`:
    - if contains `"Cr"`, call `convert_cr_to_lacs`
    - elif contains `"$"`, call `convert_dollar_to_lacs`
    - else clean up
     - strip commas, quotes, `₹`, `P.A.`, `Lacs`
     - parse hyphen range
     - if numeric and <500, multiply by 100000 (to lacs→annual INR)
     - return `"low-high"` (or same for fixed)
    - non-string: `"Not Disclosed"`

3. Apply to `df["salary"]`:
  - `df["salary"] = df["salary"].apply(convert_salary_value)`

4. Export cleaned values:
  - `df["salary"].to_csv("cleaned_salaries.csv", index=False)`

In [127]:
#convert salary values to lacs
def convert_dollar_to_lacs(salary):
  if pd.isna(salary) or not isinstance(salary, str):
    return salary
  salary = salary.replace("$", "").strip()
  salary = salary.replace(",", "")
  is_above = "and above" in salary
  numbers = re.findall(r"\d+\.?\d*", salary)
  numbers_lacs = [float(num) * 80 for num in numbers]
  if len(numbers_lacs) == 2:
    result = f"{numbers_lacs[0]}-{numbers_lacs[1]}"
  elif len(numbers_lacs) == 1:
    result = f"{numbers_lacs[0]}"
  else:
    result = salary
  if result== "":
    print(salary)
  return result
#convert salary values in crores to lacs
def convert_cr_to_lacs(salary):
  if pd.isna(salary) or not isinstance(salary, str):
    return salary
  salary = salary.replace("\u20B9", "").strip()
  is_above = "and above" in salary
  numbers = re.findall(r"\d+\.?\d*", salary)
  numbers_lacs = [float(num) * 10000000 for num in numbers]
  if len(numbers_lacs) == 2:
    result = f"{numbers_lacs[0]}-{numbers_lacs[1]}"
  elif len(numbers_lacs) == 1:
    result = f"{numbers_lacs[0]}"
  else:
    result = salary
  return result

def convert_salary_value(x):
    if isinstance(x, str) and "Not Disclosed" in x:
        return "0-0"
    if isinstance(x, str) and "Cr" in x:
        return convert_cr_to_lacs(x)
    elif isinstance(x, str) and "$" in x:
        return convert_dollar_to_lacs(x)
    elif isinstance(x, str):
        cleanrange = x.replace(',', '').strip().replace('"', '').strip().replace("₹", "").strip().replace("P.A.", "").strip().replace("Lacs", "").strip().split('(')[0].strip()
        if cleanrange.find('-') != -1:
            parts = cleanrange.split('-')
            if len(parts) == 2:
                try:
                  low = float(parts[0])
                  high = float(parts[1])
                  if low < 500:
                    low *= 100000
                  if high < 500:
                    high *= 100000
                  cleanrange = f"{low}-{high}"
                except ValueError:
                  cleanrange = f"{parts[0]}-{parts[1]}"
            elif len(parts) == 1:
              cleanrange = f"{parts[0]}"
        else:
            try:
              value = float(cleanrange)
              if value < 500:
                value *= 100000
              cleanrange = f"{value}-{value}"
            except ValueError:
              cleanrange = f"{cleanrange}"
        return cleanrange
    else:
        return "0-0"
def clean_experience(x):
    if pd.isna(x) or not isinstance(x, str):
        return x
    x = x.strip()
    if x.lower() in ["not disclosed", "not specified", "not mentioned"]:
        return "Not Disclosed"
    numbers = re.findall(r"\d+\.?\d*", x)
    if len(numbers) == 2:
        return f"{numbers[0]}-{numbers[1]}"
    elif len(numbers) == 1:
        return f"{numbers[0]}-{numbers[0]}"
    else:
        return x

`exp_req` is normalized by `clean_experience`:

- if missing/NaN or not string: kept as-is.
- if `"not disclosed"`/`"not specified"`/`"not mentioned"` (case-insensitive): converted to `"Not Disclosed"`.
- otherwise extract all numeric tokens using regex `\d+\.?\d*`.
- if two numbers found: set `"min-max"` (e.g. `3-5` stays `3-5`).
- if one number found: duplicate as range `"x-x"` (e.g. `4` -> `4-4`).
- if none, leave original text.

Then:
- `df["exp_req"]` set to string type.
- split into two columns:
  - `min_exp` (before `-`)
  - `max_exp` (after `-`)

In [128]:

df["salary"] = df["salary"].apply(convert_salary_value).astype(str)
df[["min_salary", "max_salary"]] = df["salary"].str.split("-", expand=True)
df["min_salary"] = df["min_salary"].astype(float, errors='ignore').fillna(0).astype(float)
df["max_salary"] = df["max_salary"].astype(float, errors='ignore').fillna(0).astype(float)
df["exp_req"] = df["exp_req"].apply(clean_experience).astype(str)
df[["min_exp", "max_exp"]] = df["exp_req"].str.split("-", expand=True)
df["min_exp"] = df["min_exp"].astype(int, errors='ignore').fillna(0).astype(int)
df["max_exp"] = df["max_exp"].astype(int, errors='ignore').fillna(0).astype(int)
df["working_time"] = df["emp_type"].str.split(',').str[0].str.strip()
df["contract_type"] = df["emp_type"].str.split(',').str[1].str.strip()
df["applicants_count"] = df["applicants"].apply(lambda x: re.search(r"\d+", str(x)).group() if re.search(r"\d+", str(x)) else x,)
df["applicants_count"] = df["applicants_count"].astype(int, errors='ignore').fillna(0).astype(int)
df["location_count"] = df["location"].apply(lambda x: len(eval(x)) if isinstance(x, str) else 0).astype(int)
df["key_skills_count"] = df["key_skills"].apply(lambda x: len(eval(x)) if isinstance(x, str) else 0).astype(int)


In [129]:
# import matplotlib.pyplot as plt
# from scipy.stats import norm
# df["rating"]

# # Drop missing values
# data = df["rating"].dropna()

# # Plot histogram (normalized)
# plt.hist(data, bins=30, density=True)

# # Fit normal distribution
# mu, std = norm.fit(data)

# # Create normal curve
# xmin, xmax = plt.xlim()
# x = np.linspace(xmin, xmax, 100)
# y = norm.pdf(x, mu, std)

# # Plot curve
# plt.plot(x, y)

# # Title
# plt.title(f"Rating Distribution (mean={mu:.2f}, std={std:.2f})")

# plt.show()


In [130]:

#since the rating column mean is exactly falling at 50% and more often in data hence I replace the NaN values with mean of the column to avoid biasing the data with 0 which is not the case in this dataset
meanRating = df["rating"].mean()
#imputation of missing values in rating column with mean of the column to avoid biasing the data with 0 which is not the case in this dataset
df["rating"] = df["rating"].astype(float, errors='ignore').fillna(meanRating).astype(float)

In [131]:
#implementation for the MLP 
#sorting the columns based on their data types to make it easier for the MLP model to process the data
df = df.reindex(sorted(df.columns, key=lambda x: str(df[x].dtype)), axis=1)

In [132]:
df_mlp= df[["min_salary", "max_salary", "min_exp", "max_exp", "location_count", "key_skills_count", "applicants_count", "rating"]]
df_mlp.describe()

,min_salary,max_salary,min_exp,max_exp,location_count,key_skills_count,applicants_count,rating
count,1.879000e+04,1.879000e+04,18790.000000,18790.000000,18790.000000,18790.000000,18790.000000,18790.000000
mean,1.714633e+05,2.625679e+05,3.274401,6.640607,2.466312,8.098563,430.844119,3.986749
std,1.390357e+06,1.453277e+06,3.119143,3.911551,2.872846,4.021348,1956.484638,0.412681
min,0.000000e+00,0.000000e+00,0.000000,0.000000,1.000000,1.000000,0.000000,1.000000
25%,0.000000e+00,0.000000e+00,0.000000,4.000000,1.000000,5.000000,10.000000,3.800000
50%,0.000000e+00,0.000000e+00,3.000000,6.000000,1.000000,9.000000,36.000000,4.000000
75%,1.000000e+05,2.500000e+05,5.000000,9.000000,3.000000,10.000000,220.000000,4.200000
max,9.500000e+07,9.500000e+07,25.000000,31.000000,69.000000,39.000000,88785.000000,5.000000


In [133]:
import numpy as np

class MLP:
    def __init__(self, architecture, learning_rate=0.01, epochs=1000):
        self.architecture = architecture
        self.learning_rate = learning_rate
        self.epochs = epochs
        self.parameters = {}
        self.loss_history = []

        self.initialize_parameters()

    def initialize_parameters(self):
        np.random.seed(42)
        for l in range(1, len(self.architecture)):
            self.parameters['W' + str(l)] = np.random.randn(
                self.architecture[l], self.architecture[l-1]
            ) * 0.01
            self.parameters['b' + str(l)] = np.zeros((self.architecture[l], 1))

    def relu(self, Z):
        return np.maximum(0, Z)

    def relu_derivative(self, Z):
        return Z > 0

    def sigmoid(self, Z):
        return 1 / (1 + np.exp(-Z))

    def forward_propagation(self, X):
        cache = {}
        A = X.T  # shape: (features, samples)

        for l in range(1, len(self.architecture) - 1):
            Z = np.dot(self.parameters['W' + str(l)], A) + self.parameters['b' + str(l)]
            A = self.relu(Z)
            cache['Z' + str(l)] = Z
            cache['A' + str(l)] = A

        # Output layer
        L = len(self.architecture) - 1
        ZL = np.dot(self.parameters['W' + str(L)], A) + self.parameters['b' + str(L)]
        AL = self.sigmoid(ZL)  # can change based on task

        cache['Z' + str(L)] = ZL
        cache['A' + str(L)] = AL

        return AL, cache

    def compute_loss(self, Y, AL):
        m = Y.shape[0]
        Y = Y.reshape(1, -1)
        loss = - (1/m) * np.sum(Y * np.log(AL + 1e-8) + (1 - Y) * np.log(1 - AL + 1e-8))
        return loss

    def backward_propagation(self, X, Y, cache):
        grads = {}
        m = X.shape[0]
        Y = Y.reshape(1, -1)

        L = len(self.architecture) - 1
        AL = cache['A' + str(L)]

        # Output layer gradient
        dZ = AL - Y

        for l in reversed(range(1, L + 1)):
            A_prev = X.T if l == 1 else cache['A' + str(l-1)]

            grads['dW' + str(l)] = (1/m) * np.dot(dZ, A_prev.T)
            grads['db' + str(l)] = (1/m) * np.sum(dZ, axis=1, keepdims=True)

            if l > 1:
                Z_prev = cache['Z' + str(l-1)]
                dA_prev = np.dot(self.parameters['W' + str(l)].T, dZ)
                dZ = dA_prev * self.relu_derivative(Z_prev)

        return grads

    def update_parameters(self, grads):
        for l in range(1, len(self.architecture)):
            self.parameters['W' + str(l)] -= self.learning_rate * grads['dW' + str(l)]
            self.parameters['b' + str(l)] -= self.learning_rate * grads['db' + str(l)]

    def fit(self, X, Y):
        for epoch in range(self.epochs):
            AL, cache = self.forward_propagation(X)
            loss = self.compute_loss(Y, AL)
            grads = self.backward_propagation(X, Y, cache)
            self.update_parameters(grads)

            self.loss_history.append(loss)

            if epoch % 100 == 0:
                print(f"Epoch {epoch}, Loss: {loss:.4f}")

    def predict(self, X):
        AL, _ = self.forward_propagation(X)
        return (AL > 0.5).astype(int)

In [134]:
# Features
df_mlp = df[[
    "min_salary", "max_salary", "min_exp", "max_exp",
    "location_count", "key_skills_count",
    "applicants_count", "rating"
]]

X = df_mlp.values
y = df_mlp["rating"].values  # replace with your target column

# Normalize (IMPORTANT)
X = (X - X.mean(axis=0)) / X.std(axis=0)

# Define architecture
# 8 inputs → 2 hidden layers → 1 output
mlp = MLP(architecture=[8, 16, 8, 1], learning_rate=0.01, epochs=1000)

# Train
mlp.fit(X, y)

# Predict
predictions = mlp.predict(X)

Epoch 0, Loss: 0.6932
Epoch 100, Loss: -9.5683
Epoch 200, Loss: -50.9089
Epoch 300, Loss: nan


/var/folders/qc/kkzd92996fl0r26bx2qcwvlc0000gn/T/ipykernel_49889/2077526737.py:76: RuntimeWarning: invalid value encountered in multiply
  dZ = dA_prev * self.relu_derivative(Z_prev)


Epoch 400, Loss: nan
Epoch 500, Loss: nan
Epoch 600, Loss: nan
Epoch 700, Loss: nan
Epoch 800, Loss: nan
Epoch 900, Loss: nan
